# Customer Feedback Analysis and Automated Response

**Imarticus Data Science Internship Assessment**

## Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from collections import Counter

## Loading and Reading the Dataset

In [2]:
df = pd.read_csv("Mobile Reviews Sentiment.csv")

In [3]:
df.head()

,review_id,customer_name,age,brand,model,price_usd,price_local,currency,exchange_rate_to_usd,rating,...,verified_purchase,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,review_length,word_count,helpful_votes,source
0,1,Aryan Maharaj,45,Realme,Realme 12 Pro,337.31,₹27996.73,INR,83.00,2,...,True,1,1,3,2,1,46,7,1,Amazon
1,2,Davi Miguel Sousa,18,Realme,Realme 12 Pro,307.78,R$1754.35,BRL,5.70,4,...,True,3,2,4,3,2,74,12,5,Flipkart
2,3,Pahal Balay,27,Google,Pixel 6,864.53,₹71755.99,INR,83.00,4,...,True,3,5,3,2,4,55,11,8,AliExpress
3,4,David Guzman,19,Xiaomi,Redmi Note 13,660.94,د.إ2425.65,AED,3.67,3,...,False,1,3,2,1,2,66,11,3,Amazon
4,5,Yago Leão,38,Motorola,Edge 50,792.13,R$4515.14,BRL,5.70,3,...,True,3,3,2,2,1,73,12,0,BestBuy


In [4]:
df.shape

(50000, 25)

## Data Quality Assesment

### 1. Checking for Duplicates

In [5]:
df.duplicated().sum()

np.int64(0)

### 2. Checking for Missing Values

In [6]:
df.isnull().sum()

review_id               0
customer_name           0
age                     0
brand                   0
model                   0
price_usd               0
price_local             0
currency                0
exchange_rate_to_usd    0
rating                  0
review_text             0
sentiment               0
country                 0
language                0
review_date             0
verified_purchase       0
battery_life_rating     0
camera_rating           0
performance_rating      0
design_rating           0
display_rating          0
review_length           0
word_count              0
helpful_votes           0
source                  0
dtype: int64

## Cleaning Review Text

In [7]:
df['review_text'] = [''.join(char for char in text.lower() if char.isalpha() or char.isspace()) 
                     for text in df['review_text']]
df['review_text'] = df['review_text'].str.lower().str.strip()

In [8]:
df['review_text'].head()

0          not worth the money spent wouldnt recommend
1    absolutely love this phone the camera is next ...
2    loving the clean ui and fast updates loving it...
3    build quality feels solid and durable no regre...
4    not bad for daily use but could be optimized a...
Name: review_text, dtype: object

## Filtering Critical Reviews

In [9]:
# Filter reviews where rating <= 2, sentiment is 'negative', and purchase is verified
critical_df = df[(df['rating'] <= 2) & (df['sentiment'] == 'Negative') & (df['verified_purchase'] == True)].copy()

print(f"Total Critical Verified Negative Reviews Isolated: {len(critical_df)}")

critical_df[['rating', 'sentiment', 'verified_purchase', 'review_text']].head()

Total Critical Verified Negative Reviews Isolated: 7117


,rating,sentiment,verified_purchase,review_text
0,2,Negative,True,not worth the money spent wouldnt recommend
7,2,Negative,True,phone hangs often regret buying it wouldnt rec...
30,1,Negative,True,customer service was disappointing wouldnt rec...
31,1,Negative,True,charging is very slow compared to other brands...
32,2,Negative,True,speaker quality is bad and muffled returning t...


## Defining Function for Top Complaint Keywords

In [10]:
def top_critical_keywords(review_series, keywords_list):
    found_keywords = []

    for review in review_series:
        review_str = str(review)
        for keyword in keywords_list:
            if keyword in review_str:
                found_keywords.append(keyword)
                
    top_3_counts = Counter(found_keywords).most_common(3)
    
    print("--- Top 3 Complaint Keywords Found ---")
    for keyword, count in top_3_counts:
        print(f"'{keyword}': found {count} times")
        
    return [item[0] for item in top_3_counts]

target_keywords = ["hangs", "lags", "speaker", "customer service", "overheats", "battery", "charging", "display"]

top_3_complaints = top_critical_keywords(critical_df['review_text'], target_keywords)

--- Top 3 Complaint Keywords Found ---
'overheats': found 747 times
'lags': found 740 times
'customer service': found 735 times


## Automated Outreach (Generative AI)

In this section, we select 3 of the most critical, detailed negative reviews from our filtered dataset and pass them through a Generative AI API to automatically generate highly personalized, empathetic customer support apology emails.

In [11]:
!pip install google-genai

Defaulting to user installation because normal site-packages is not writeable


In [12]:
import os
from google import genai

In [13]:
client = genai.Client(api_key="Insert API Key")

print("Gen AI Client successfully initialized and ready!")

Gen AI Client successfully initialized and ready!


In [14]:
review_1 = critical_df[critical_df['review_text'].str.contains('overheat', case=False)].head(1)
review_2 = critical_df[critical_df['review_text'].str.contains('lags', case=False)].head(1)
review_3 = critical_df[critical_df['review_text'].str.contains('customer service', case=False)].head(1)

sample_reviews = pd.concat([review_1, review_2, review_3])

for review in sample_reviews['review_text'].values:
    print("Review Text:")
    print(review)
    print("-" * 50)

Review Text:
overheats quickly while gaming very disappointed
--------------------------------------------------
Review Text:
phone lags often after a few apps very disappointed
--------------------------------------------------
Review Text:
customer service was disappointing wouldnt recommend
--------------------------------------------------


In [17]:
# Loop through our 3 selected reviews using a standard for loop
for review in sample_reviews['review_text'].values:
    
    # Craft the clear prompt for the AI agent
    prompt = f"""
    You are a professional Customer Support Agent. 
    Write a short, personalized, and deeply empathetic apology email addressing the specific complaints mentioned in this customer review.
    
    Customer Review:
    "{review}"
    
    Ensure the email:
    - Directly acknowledges their specific issue.
    - Offers a polite, reassuring tone.
    """
    
    # FIXED: Swapped out the retired model for the current active production model
    response = client.models.generate_content(
        model='gemini-3.5-flash', 
        contents=prompt,
    )
    
    # Print the results clearly in your notebook
    print("==================================================")
    print(f"ORIGINAL CUSTOMER COMPLAINT:\n{review}\n")
    print(f"GENERATED AI APOLOGY EMAIL:\n{response.text}")
    print("==================================================\n")

ORIGINAL CUSTOMER COMPLAINT:
overheats quickly while gaming very disappointed

GENERATED AI APOLOGY EMAIL:
Subject: We want to make your gaming experience right 

Hi [Customer Name],

I noticed your recent review about your [Product Name] overheating quickly while you are gaming, and I am so sorry for the disappointment this has caused. 

I completely understand how frustrating it is to have your gaming sessions cut short by temperature issues. When you sit down to play, you deserve a smooth, uninterrupted experience, and it's clear we fell short of that for you. Please rest assured that this is not the standard of quality we aim to deliver.

We want to make this right for you immediately. Please reply directly to this email so we can assist you with some quick troubleshooting steps, or arrange a hassle-free replacement or refund. 

We are here to support you, and we won't rest until you are completely satisfied with your experience. 

Warm regards,

[Your Name]  
Customer Support Team

## Output Artifacts

### Calculated Insights

Based on our text analysis and counting function applied to the filtered dataset (`critical_df`), the top 3 most common keywords driving negative customer sentiment are:

| Rank | Keyword | Issue Category |
| :--- | :--- | :--- |
| **1** | `overheat` | Hardware / Thermal Regulation |
| **2** | `lags` | Software Performance / Optimization |
| **3** | `customer service` | Post-Purchase Support Experience |

### AI-Generated Response Emails

Here are the automated outreach emails generated by the Gemini model addressing our specific customer pain points:

### Email 1: Overheating Issue
* **Original Customer Complaint:** "overheats quickly while gaming very disappointed"
* **Generated AI Apology Email:**
  > **Subject:** We want to make your gaming experience right 
  >
  > Hi [Customer Name],
  >
  > I noticed your recent review about your [Product Name] overheating quickly while you are gaming, and I am so sorry for the disappointment this has caused. 
  >
  > I completely understand how frustrating it is to have your gaming sessions cut short by temperature issues. When you sit down to play, you deserve a smooth, uninterrupted experience, and it's clear we fell short of that for you. Please rest assured that this is not the standard of quality we aim to deliver.
  >
  > We want to make this right for you immediately. Please reply directly to this email so we can assist you with some quick troubleshooting steps, or arrange a hassle-free replacement or refund. 
  >
  > We are here to support you, and we won't rest until you are completely satisfied with your experience. 
  >
  > Warm regards,
  >
  > [Your Name]  
  > Customer Support Team  
  > [Company Name]

---

### Email 2: Device Lagging Issue
* **Original Customer Complaint:** "phone lags often after a few apps very disappointed"
* **Generated AI Apology Email:**
  > **Subject:** We're so sorry your phone is lagging – let’s get this fixed
  >
  > Hi [Customer Name],
  >
  > Thank you for sharing your feedback with us. I am so incredibly sorry to hear that your phone has been lagging when you have just a few apps open. I completely understand how disappointing and frustrating it is to deal with a slow device when you just need it to work smoothly. 
  >
  > Please rest assured that this is not the standard we strive for, and we want to make things right. You deserve a phone that can easily keep up with your daily routine. 
  >
  > I would love to help you resolve this as quickly as possible. Could you please reply to this email with your phone model and order details? Once I have those, I can walk you through some quick steps to boost your phone's performance, or we can discuss replacement options. 
  >
  > Thank you so much for your patience. I look forward to getting this sorted out for you!
  >
  > Warmly,
  >
  > [Your Name]  
  > Customer Support Team

---

### Email 3: Customer Service Experience
* **Original Customer Complaint:** "customer service was disappointing wouldnt recommend"
* **Generated AI Apology Email:**
  > **Subject:** Sincere apologies regarding your customer service experience
  >
  > Dear [Customer Name],
  >
  > I recently read your review about your disappointing experience with our customer service team, and I am truly sorry that we let you down. Hearing that you wouldn't recommend us is incredibly tough to hear, as we always aim to provide a warm, helpful, and supportive experience. 
  >
  > Please rest assured that this is not the standard we hold ourselves to. Your feedback is being taken very seriously, and I would love the opportunity to make things right. 
  >
  > If you are open to sharing, could you please reply directly to this email with a few more details about what happened? I want to personally investigate this, address your concerns, and ensure we do better in the future. 
  >
  > Thank you for your honesty, and I hope we can have the chance to earn back your trust.
  >
  > Warmly,
  >
  > [Your Name]  
  > [Your Title]  
  > [Company Name]